In [ ]:
import os
import sys
import pandas as pd
from src.utils.db_utils import get_connection, execute_query
from src.utils.schedule_utility import get_full_schedule, get_current_week, get_team_schedule

## Bye Weeks
def byeweeks():
    """Bye weeks """
    query = """
    select *
    from stats.byeweek
    where season = '2024'
    """
    return  execute_query(query)

bye_weekdf = byeweeks()

print("Bye Week")
print(bye_weekdf)



In [ ]:
## Tean Stats Joined w/ Game Summary
def get_teamstats_with_game_context(gamesummaryid=None, season=None):
    """Get teamstats with game context fields populated"""
    
    # Base query with the CASE logic
    query = """
    SELECT 
        ts.season,
        ts.week,
        ts.gamesummaryid,
        ts.teamid,
        ts.hometeamid,
        ts.awayteamid,
        ts.total_yards,
        -- Game context fields based on home/away logic
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.days_since_last_game_hometeam
            WHEN ts.teamid = gs.awayteamid THEN gs.days_since_last_game_awayteam
            ELSE NULL
        END as days_since_last_game,
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.hometeam_weeks_from_bye
            WHEN ts.teamid = gs.awayteamid THEN gs.awayteam_weeks_from_bye
            ELSE NULL
        END as weeks_from_bye
    FROM stats.teamstats ts
    LEFT JOIN stats.gamesummary gs ON ts.gamesummaryid = gs.gamesummaryid
    """
    
    # Add WHERE conditions based on parameters
    where_conditions = []
    if gamesummaryid:
        where_conditions.append(f"ts.gamesummaryid = '{gamesummaryid}'")
    if season:
        where_conditions.append(f"ts.season = {season}")
    
    if where_conditions:
        query += " WHERE " + " AND ".join(where_conditions)
    
    query += " ORDER BY ts.gamesummaryid, ts.teamid"
    
    return execute_query(query)

print(" ")
print("Team Stats joine w/ GameSummary - Weeks since/from Bye week, Days since last game")
df_test = get_teamstats_with_game_context(gamesummaryid='gs-202410DETHOU')
#print("Test game results:")
#print(df_test)


In [10]:
## Game Weather

def gameweather():
    """Bye weeks """
    query = """
    select gamesummaryid, hometeamid, awayteamid, kickoff_temperature_f, kickoff_wind_speed_mph, kickoff_wind_direction_deg, kickoff_humidity_pct, kickoff_weather_description,
    kickoff_pressure_mb, 
    game_total_rain_in, game_total_snow_in
    from stats.gameweather
    limit 1
    """
    return  execute_query(query)

game_weather = gameweather()

print(" ")
print("Game Weather")
print(game_weather)

Successfully connected to the database!
 
Game Weather
     gamesummaryid hometeamid awayteamid  kickoff_temperature_f  \
0  gs-202401BALKAN        KAN        BAL                   76.3   

   kickoff_wind_speed_mph  kickoff_wind_direction_deg  kickoff_humidity_pct  \
0                     4.2                          32                    74   

  kickoff_weather_description  kickoff_pressure_mb  game_total_rain_in  \
0                Mainly clear                989.5                 0.0   

   game_total_snow_in  
0                 0.0  


In [17]:
## Schedule utility

full_schedule = get_full_schedule()
print("Full Schedule")
print(full_schedule)

curent_games = get_current_week()
print(" ")
print("Current Week")
print(curent_games)


phi_schedule = get_team_schedule('PHI')
print(" ")
print("Eagles Schedule")
print(phi_schedule)


Successfully connected to the database!
✅ Retrieved 272 total games
Full Schedule
     week  day        date hometeam awayteam
0       1  Thu  2025-09-04      PHI      DAL
1       1  Fri  2025-09-05      LAC      KAN
2       1  Sun  2025-09-07      NYJ      PIT
3       1  Sun  2025-09-07      IND      MIA
4       1  Sun  2025-09-07      WAS      NYG
..    ...  ...         ...      ...      ...
267    18  Sun  2025-01-04      NWE      MIA
268    18  Sun  2025-01-04      MIN      GNB
269    18  Sun  2025-01-04      JAX      TEN
270    18  Sun  2025-01-04      DEN      LAC
271    18  Sun  2025-01-04      HOU      IND

[272 rows x 5 columns]
Successfully connected to the database!
⚠️  No games found for current week
 
Current Week
Empty DataFrame
Columns: [week, day, date, hometeam, awayteam]
Index: []
Successfully connected to the database!
🏈 Retrieved 17 games for PHI
 
Eagles Schedule
    week  day        date hometeam awayteam home_away opponent
0      1  Thu  2025-09-04      PHI      